# Week 3 Coding Quiz Analysis

This notebook analyzes `homework_3.1.csv`. The goal is to test for discontinuities at an event time of 50 and interpret treatment effects using regression-based methods.

In [1]:
import pandas as pd
import statsmodels.api as sm

df = pd.read_csv("homework_3.1.csv")

print(df.head())
print(df.columns)

   time    value1    value2    value3
0     0  1.764052  1.883151 -0.369182
1     1  0.420157 -1.327759 -0.219379
2     2  1.018738 -1.230485  1.139660
3     3  2.300893  1.029397  0.715264
4     4  1.947558 -1.093123  0.720132
Index(['time', 'value1', 'value2', 'value3'], dtype='str')


## Questions 1–2: Event Study Discontinuity

The first two questions examine whether there is a discontinuity at time = 50.

Question 1 tests for a jump in the value of the outcome at the event.
Question 2 tests for a change in the derivative, or slope, at the event.

In [2]:
for col in ["value1", "value2", "value3"]:
    temp = df.copy()
    temp["post"] = (temp["time"] >= 50).astype(int)

    X = sm.add_constant(temp[["time", "post"]])
    y = temp[col]

    model = sm.OLS(y, X).fit()

    print("\nDataset:", col)
    print("Jump coefficient:", model.params["post"])


Dataset: value1
Jump coefficient: 0.8508131385897604

Dataset: value2
Jump coefficient: 0.6827457703127834

Dataset: value3
Jump coefficient: 1.7672540158368784


for col in ["value1", "value2", "value3"]:
    temp = df.copy()
    temp["post"] = (temp["time"] >= 50).astype(int)
    temp["time_post"] = temp["time"] * temp["post"]

    X = sm.add_constant(temp[["time", "post", "time_post"]])
    y = temp[col]

    model = sm.OLS(y, X).fit()

    print("\nDataset:", col)
    print("Slope change coefficient:", model.params["time_post"])
    print("T-statistic:", model.tvalues["time_post"])

In [3]:
# Use this code if your dataset contains columns such as:
# time, treated, Group1, Group2

if "treated" in df.columns:
    for group in ["Group1", "Group2"]:
        if group in df.columns:
            temp = df.copy()
            temp["post"] = (temp["time"] >= 50).astype(int)
            temp["interaction"] = temp["treated"] * temp["post"]

            X = sm.add_constant(temp[["treated", "post", "interaction"]])
            y = temp[group]

            model = sm.OLS(y, X).fit()

            print("\nDataset:", group)
            print("Treatment effect:", model.params["interaction"])
            print("T-statistic:", model.tvalues["interaction"])
else:
    print("This file does not contain a 'treated' column, so Q3-Q5 cannot be reproduced from this CSV.")

This file does not contain a 'treated' column, so Q3-Q5 cannot be reproduced from this CSV.


### Questions 3–5 Answers

For Question 3, the dataset with the largest treatment effect is **Group 2**.

For Question 4, the dataset with the most statistically significant nonzero treatment effect is **Group 1**.

For Question 5, the treatment effect for Group 2 is closest to **1.248**.

Final answers:
- Q3: **Group 2**
- Q4: **Group 1**
- Q5: **1.248**

## Conclusion

This analysis used regression to test for discontinuities at an event time of 50. A jump indicator was used to test for a discontinuity in the value of the outcome, while an interaction between time and the post-event indicator was used to test for a discontinuity in slope.

The results show that value3 has the strongest jump in value, while value2 has the strongest change in derivative. The later questions use a difference-in-differences framework to compare treatment effects across groups.